In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display

# ==========================================
# 1. Loss Function & Gradient Definitions
# ==========================================

def rosenbrock(x: np.ndarray, a: float = 1.0, b: float = 100.0) -> float:
    """Rosenbrock function: f(x, y) = (a - x)^2 + b * (y - x^2)^2"""
    return (a - x[0]) ** 2 + b * (x[1] - x[0] ** 2) ** 2


def rosenbrock_grad(x: np.ndarray, a: float = 1.0, b: float = 100.0) -> np.ndarray:
    """Gradient vector of the Rosenbrock function."""
    df_dx = -2 * (a - x[0]) - 4 * b * x[0] * (x[1] - x[0] ** 2)
    df_dy = 2 * b * (x[1] - x[0] ** 2)
    return np.array([df_dx, df_dy])


# ==========================================
# 2. Optimization Engine
# ==========================================

def run_optimization(optimizer_type: str, lr: float, beta: float, max_steps: int = 200):
    init_x = np.array([-1.2, 1.0])
    x = init_x.copy().astype(np.float64)
    v = np.zeros_like(x)

    trajectory = [x.copy()]
    losses = [rosenbrock(x)]
    grad_norms = [np.linalg.norm(rosenbrock_grad(x))]

    for _ in range(max_steps):
        grad = rosenbrock_grad(x)
        
        if optimizer_type == "Standard GD":
            x -= lr * grad
        elif optimizer_type == "Momentum":
            v = beta * v + lr * grad
            x -= v

        trajectory.append(x.copy())
        losses.append(rosenbrock(x))
        grad_norms.append(np.linalg.norm(grad))

    return np.array(trajectory), np.array(losses), np.array(grad_norms)


# ==========================================
# 3. Visualization Dashboard
# ==========================================

def update_visualizer(optimizer_type, lr, beta, current_step):
    # Execute optimization based on current slider values
    trajectory, losses, grad_norms = run_optimization(optimizer_type, lr, beta)
    
    # Create subplots layout
    fig = plt.figure(figsize=(16, 5))
    ax_contour = fig.add_subplot(131)
    ax_loss = fig.add_subplot(132)
    ax_grad = fig.add_subplot(133)

    # --- Plot 1: Contour & Animated Trajectory ---
    X = np.linspace(-1.5, 1.5, 150)
    Y = np.linspace(-0.5, 1.5, 150)
    X_grid, Y_grid = np.meshgrid(X, Y)
    Z_grid = np.array([rosenbrock(np.array([x, y])) for x, y in zip(X_grid.ravel(), Y_grid.ravel())]).reshape(X_grid.shape)

    ax_contour.contour(X_grid, Y_grid, Z_grid, levels=np.logspace(-1, 3, 20), cmap="magma", alpha=0.6)
    
    # Trajectory path up to current step
    traj_sub = trajectory[:current_step + 1]
    ax_contour.plot(traj_sub[:, 0], traj_sub[:, 1], "o-", color="#1D3557", markersize=3, alpha=0.7)
    ax_contour.plot(trajectory[0, 0], trajectory[0, 1], "go", markersize=8, label="Start Point")
    ax_contour.plot(traj_sub[-1, 0], traj_sub[-1, 1], "ro", markersize=8, label="Current Position")
    ax_contour.plot(1.0, 1.0, "k*", markersize=12, label="Global Min (1,1)")

    ax_contour.set_title(f"2D Optimization Path (Step {current_step})")
    ax_contour.set_xlabel("$x_1$")
    ax_contour.set_ylabel("$x_2$")
    ax_contour.legend(loc="upper left")
    ax_contour.grid(True, linestyle="--", alpha=0.5)

    # --- Plot 2: Loss Curve ---
    ax_loss.plot(losses[:current_step + 1], color="#E63946", linewidth=2)
    ax_loss.set_xlim(0, len(losses))
    ax_loss.set_yscale("log")
    ax_loss.set_title(f"Loss Value: {losses[current_step]:.4e}")
    ax_loss.set_xlabel("Step")
    ax_loss.set_ylabel("Loss $\mathcal{L}(x)$ [Log Scale]")
    ax_loss.grid(True, linestyle="--", alpha=0.5)

    # --- Plot 3: Gradient Norm Curve ---
    ax_grad.plot(grad_norms[:current_step + 1], color="#2A9D8F", linewidth=2)
    ax_grad.set_xlim(0, len(grad_norms))
    ax_grad.set_yscale("log")
    ax_grad.set_title(f"Gradient Norm: {grad_norms[current_step]:.4e}")
    ax_grad.set_xlabel("Step")
    ax_grad.set_ylabel("$\|\| \\nabla f(x) \|\|$ [Log Scale]")
    ax_grad.grid(True, linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.show()


# ==========================================
# 4. Interactive Widget Setup
# ==========================================

optimizer_dropdown = widgets.Dropdown(
    options=["Standard GD", "Momentum"],
    value="Momentum",
    description="Optimizer:",
)

lr_slider = widgets.FloatLogSlider(
    value=0.001,
    base=10,
    min=-4,
    max=-1,
    step=0.1,
    description="Learning Rate:",
    style={'description_width': 'initial'}
)

beta_slider = widgets.FloatSlider(
    value=0.9,
    min=0.0,
    max=0.99,
    step=0.01,
    description="Momentum (β):",
    style={'description_width': 'initial'}
)

step_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=200,
    step=1,
    description="Step / Frame:",
    style={'description_width': 'initial'}
)

# Connect UI control elements
widgets.interactive_output(
    update_visualizer,
    {
        "optimizer_type": optimizer_dropdown,
        "lr": lr_slider,
        "beta": beta_slider,
        "current_step": step_slider,
    }
)

# Render Controls Layout
controls = widgets.VBox([
    widgets.HBox([optimizer_dropdown, lr_slider, beta_slider]),
    step_slider
])

display(controls)
widgets.interactive_output(update_visualizer, {
    "optimizer_type": optimizer_dropdown,
    "lr": lr_slider,
    "beta": beta_slider,
    "current_step": step_slider
})